In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

wnut = load_dataset("wnut_17", trust_remote_code=True)

ner_tags = wnut["train"].features["ner_tags"].feature
id2label = {i: tag for i, tag in enumerate(ner_tags.names)}
label2id = {tag: i for i, tag in enumerate(ner_tags.names)}

print("id2label:", id2label)
print("label2id:", label2id)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        padding="max_length",
        is_split_into_words=True,
    )

    labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label = examples["ner_tags"][i]
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = wnut.map(preprocess_function, batched=True)

In [ ]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, label in zip(predictions, labels):
        true_labels_batch = []
        true_predictions_batch = []
        for p_, l_ in zip(pred, label):
            if l_ != -100:
                true_labels_batch.append(id2label[l_])
                true_predictions_batch.append(id2label[p_])
        true_labels.append(true_labels_batch)
        true_predictions.append(true_predictions_batch)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return results

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

args = TrainingArguments(
    output_dir="./bert-ner-wnut17",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./bert-ner-wnut17")
tokenizer.save_pretrained("./bert-ner-wnut17")
trainer.evaluate(tokenized_datasets["test"])

In [ ]:
from transformers import pipeline
import spacy
from spacy.tokens import Doc
from spacy import displacy

text = (
    "The Golden State Warriors are an American professional basketball team based in San Francisco.\n"
    "Donald Trump is current president of USA.\n"
    "Apple sells iPhone, Google is the owner of Youtube\n"
    "and Nirvana was an American rock band formed in Aberdeen, Washington.\n"
    "John Paul II was head of the Catholic Church and sovereign of the Vatican City State.\n"
)

classifier = pipeline("ner", model="./bert-ner-wnut17", aggregation_strategy="simple")
entities = classifier(text)

nlp = spacy.blank("en")
doc = nlp.make_doc(text)
spans = []

for ent in entities:
    span = doc.char_span(ent["start"], ent["end"], label=ent["entity_group"])
    if span is not None:
        spans.append(span)

doc.ents = spans
displacy.render(doc, style="ent", jupyter=True)
# displacy.serve(doc, style="ent")  # to run the local server